# 32 candidates on Colab: train articles 13,488 to 132,583

Colab's share of the 32-candidate train split. It continues Colab's existing `train.out` on Drive (articles 0 to 13,487 are already there) and stops at article 132,583. The local PC generates articles 132,584 to 193,882; the two parts are joined later with `merge_candidates.py`.

- The run resumes from the last copy on Drive. After a disconnect or the 24-hour limit, open the notebook again and run all cells: it continues where it stopped.
- Settings match the local run: 32 candidates, diversity penalty 0.5, bfloat16 (`--bf16`). Articles already in `train.out` before this notebook was first run were generated in full precision; the progress cell prints that count, so note it down for the paper.
- **Never** run this notebook in two Colab sessions at the same time: both would write the same `train.out`.

In [ ]:
# Install condacolab. The runtime restarts after this cell, that is normal.
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# Clone the repo and mount Google Drive.
!git clone https://github.com/alvrian/BRIO.git

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

%cd BRIO

In [ ]:
# Same env setup as BRIO_32_candidates.ipynb, then the fine-tuned IndoBART and the Liputan6 source/target files.
! conda create --name env --file spec-file.txt -y
! conda run --no-capture-output -n env pip install -r requirements.txt
! conda run -n env python -c "import nltk;nltk.download('punkt')"
! git clone https://github.com/neulab/compare-mt.git
! cd compare-mt && conda run --no-capture-output -n env pip install -r requirements.txt
! cd compare-mt && conda run -n env python setup.py install
print("done setting up env")
! cp /content/drive/MyDrive/BRIO/indobart-liputan6-finetuned.zip .
! unzip -q indobart-liputan6-finetuned.zip
! rm indobart-liputan6-finetuned.zip
! cp /content/drive/MyDrive/BRIO/Dataset/liputan6_converted.zip .
! unzip -q liputan6_converted.zip
! rm liputan6_converted.zip
print("all done")

In [ ]:
# Settings. The range is fixed: Colab ends at END, where the local PC's part starts.
import os

NUM_CANDS = 32
DIVERSITY_PENALTY = 0.5
GEN_BATCH_SIZE = 4          # batch 4 was the fastest on Colab (larger batches were slower)
BF16 = True                 # bfloat16, same as the local run
START, END = 0, 132584      # start 0 = continue the existing train.out; END is exclusive (last article 132,583)
SYNC_EVERY = 200            # copy to Drive every 200 batches = 800 articles (about 45 minutes)

DATA_DIR = "./liputan6_converted/canonical"
MODEL_DIR = "./indobart-liputan6-finetuned"
CAND_DIR = "/content/drive/MyDrive/BRIO/liputan6_candidates_32/canonical"
OUT_FILE = os.path.join(CAND_DIR, "train.out")

for p in [os.path.join(DATA_DIR, "train.source"), os.path.join(DATA_DIR, "train.target"),
          os.path.join(MODEL_DIR, "config.json"), OUT_FILE]:
    print("found  " if os.path.exists(p) else "MISSING", p)

# gen_candidate.py must be the version with --start_index / --end_index
!grep -q "end_index" gen_candidate.py && echo "gen_candidate.py supports --end_index" || echo "OLD gen_candidate.py: git pull first"

In [ ]:
# Progress so far: how many articles Colab's train.out on Drive already has.
def articles_done():
    if not os.path.exists(OUT_FILE):
        return 0
    with open(OUT_FILE, encoding="utf-8") as f:
        return sum(1 for _ in f) // NUM_CANDS

done = articles_done()
print("train.out on Drive: %d articles done, %d left until article %d" % (done, END - done, END - 1))
BF16_FLAG = "--bf16" if BF16 else ""
if done > END:
    print("WARNING: train.out already goes past %d; the merge will need END to match" % END)

In [ ]:
# Generate up to article 132,583. Resumes from the last copy on Drive; run again after a disconnect.
!conda run --no-capture-output -n env python gen_candidate.py --dataset liputan6 --src_dir "{DATA_DIR}/train.source" --tgt_dir "{OUT_FILE}" --model_dir "{MODEL_DIR}" --num_cands {NUM_CANDS} --batch_size {GEN_BATCH_SIZE} --diversity_penalty {DIVERSITY_PENALTY} --start_index {START} --end_index {END} --sync_every {SYNC_EVERY} {BF16_FLAG}

In [ ]:
# Run this BEFORE stopping or restarting the session: copies the newest local working copy to Drive if it is further along,
# so the articles generated since the last copy are kept.
import shutil
local = "/content/_gen_candidate_tmp/train.out"
n_local = sum(1 for _ in open(local, encoding="utf-8")) // NUM_CANDS if os.path.exists(local) else 0
n_drive = articles_done()
print("articles in local copy: %d | on Drive: %d" % (n_local, n_drive))
if n_local > n_drive:
    shutil.copyfile(local, OUT_FILE)
    print("copied the local file to Drive")

In [ ]:
# When generation has finished: check that train.out holds exactly articles 0 to 132,583, and look for gibberish candidates.
n_lines = sum(1 for _ in open(OUT_FILE, encoding="utf-8"))
print("lines: %d, expected %d -> %s" % (n_lines, END * NUM_CANDS, "complete" if n_lines == END * NUM_CANDS else "NOT complete"))
!conda run --no-capture-output -n env python check_candidates.py --out_file "{OUT_FILE}" --tgt_file "{DATA_DIR}/train.target" --num_cands {NUM_CANDS} --show 1

### After this part is finished

1. Download `train.out` from `MyDrive/BRIO/liputan6_candidates_32/canonical/` to `D:\project\BRIO\liputan6_candidates_32\canonical\` (replacing the older copy there).
2. When the local part (`train.132584-193883.out`) is also finished, run step 5 of `BRIO_32_candidates_local.ipynb` with `SPLIT = 132584`. It checks both parts and writes `train.merged.out`.
3. Upload `train.merged.out` to the same Drive folder as `train.out`, and continue with preprocessing (step 4 of `BRIO_32_candidates.ipynb`).